In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "41b5db8d",
   "metadata": {},
   "outputs": [
    {
     "ename": "SyntaxError",
     "evalue": "unterminated string literal (detected at line 5) (3802129418.py, line 5)",
     "output_type": "error",
     "traceback": [
      "  \u001b[36mCell\u001b[39m\u001b[36m \u001b[39m\u001b[32mIn[4]\u001b[39m\u001b[32m, line 5\u001b[39m\n\u001b[31m    \u001b[39m\u001b[31mGenerate random data using Linux `/dev/urandom`, OpenSSL's\u001b[39m\n                                                            ^\n\u001b[31mSyntaxError\u001b[39m\u001b[31m:\u001b[39m unterminated string literal (detected at line 5)\n"
     ]
    }
   ],
   "source": [
    "\n",
    "# Crypto Lab Assignment 9\n",
    "## Evaluation of Operating-System Random Number Generators\n",
    "\n",
    "### Objective\n",
    "Generate random data using Linux `/dev/urandom`, OpenSSL's\n",
    "`rand` utility, and Windows CNG/.NET cryptographic APIs.\n",
    "\n",
    "Analyze byte frequencies, bit distributions, repeated patterns,\n",
    "serial correlations, and statistical tests.\n",
    "\n",
    "### Important\n",
    "Statistical tests evaluate observable properties of the generated\n",
    "data. Passing them does not prove unpredictability or cryptographic\n",
    "security.\n",
    "  "
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "f23b99e9",
   "metadata": {},
   "outputs": [],
   "source": [
    "\n",
    "from pathlib import Path\n",
    "import math\n",
    "import numpy as np\n",
    "from scipy.stats import chisquare, binomtest, norm\n",
    "\n",
    "DATA_DIR = Path(\"outputs/data\")\n",
    "files = sorted(DATA_DIR.glob(\"*.bin\"))\n",
    "\n",
    "if not files:\n",
    "    raise FileNotFoundError(\n",
    "        \"No .bin files found in outputs/data. Generate the data first.\"\n",
    "    )\n",
    "\n",
    "for file in files:\n",
    "    print(file.name, file.stat().st_size, \"bytes\")\n",
    "  "
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "d24add68",
   "metadata": {},
   "outputs": [],
   "source": [
    "\n",
    "def byte_frequency_test(data):\n",
    "    counts = np.bincount(\n",
    "        np.frombuffer(data, dtype=np.uint8),\n",
    "        minlength=256\n",
    "    )\n",
    "\n",
    "    expected = np.full(256, len(data) / 256)\n",
    "    statistic, p_value = chisquare(counts, f_exp=expected)\n",
    "\n",
    "    return counts, statistic, p_value\n",
    "\n",
    "\n",
    "def bit_balance_test(data):\n",
    "    bits = np.unpackbits(\n",
    "        np.frombuffer(data, dtype=np.uint8)\n",
    "    )\n",
    "\n",
    "    ones = int(bits.sum())\n",
    "    total = len(bits)\n",
    "\n",
    "    result = binomtest(ones, total, 0.5)\n",
    "\n",
    "    return ones, total - ones, result.pvalue\n",
    "\n",
    "\n",
    "for file in files:\n",
    "    data = file.read_bytes()\n",
    "\n",
    "    counts, chi_stat, chi_p = byte_frequency_test(data)\n",
    "    ones, zeros, bit_p = bit_balance_test(data)\n",
    "\n",
    "    print(\"\\n\", file.name)\n",
    "    print(\"Bytes:\", len(data))\n",
    "    print(\"Byte chi-square statistic:\", chi_stat)\n",
    "    print(\"Byte chi-square p-value:\", chi_p)\n",
    "    print(\"Zero bits:\", zeros)\n",
    "    print(\"One bits:\", ones)\n",
    "    print(\"Bit-balance p-value:\", bit_p)\n",
    "  "
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "f70d22dd",
   "metadata": {},
   "outputs": [],
   "source": [
    "\n",
    "def runs_test(data):\n",
    "    bits = np.unpackbits(\n",
    "        np.frombuffer(data, dtype=np.uint8)\n",
    "    ).astype(np.int8)\n",
    "\n",
    "    n0 = int(np.sum(bits == 0))\n",
    "    n1 = int(np.sum(bits == 1))\n",
    "\n",
    "    if n0 == 0 or n1 == 0:\n",
    "        return None, None\n",
    "\n",
    "    runs = 1 + int(np.count_nonzero(bits[1:] != bits[:-1]))\n",
    "\n",
    "    expected = 1 + (2 * n0 * n1) / (n0 + n1)\n",
    "\n",
    "    variance = (\n",
    "        2 * n0 * n1 * (2 * n0 * n1 - n0 - n1)\n",
    "        / (((n0 + n1) ** 2) * (n0 + n1 - 1))\n",
    "    )\n",
    "\n",
    "    if variance <= 0:\n",
    "        return runs, None\n",
    "\n",
    "    z = (runs - expected) / math.sqrt(variance)\n",
    "    p_value = 2 * norm.sf(abs(z))\n",
    "\n",
    "    return runs, p_value\n",
    "\n",
    "\n",
    "def serial_correlation(data):\n",
    "    values = np.frombuffer(data, dtype=np.uint8).astype(np.float64)\n",
    "\n",
    "    if len(values) < 2 or np.var(values) == 0:\n",
    "        return None\n",
    "\n",
    "    return float(np.corrcoef(values[:-1], values[1:])[0, 1])\n",
    "\n",
    "\n",
    "for file in files:\n",
    "    data = file.read_bytes()\n",
    "\n",
    "    runs, runs_p = runs_test(data)\n",
    "\n",
    "    print(\"\\n\", file.name)\n",
    "    print(\"Number of bit runs:\", runs)\n",
    "    print(\"Runs-test p-value:\", runs_p)\n",
    "    print(\"Serial correlation:\", serial_correlation(data))\n",
    "  "
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "1a916d14",
   "metadata": {},
   "outputs": [],
   "source": [
    "\n",
    "from collections import Counter\n",
    "import pandas as pd\n",
    "\n",
    "\n",
    "def repeated_pattern_analysis(data, block_size=4):\n",
    "    usable = len(data) - len(data) % block_size\n",
    "    blocks = [\n",
    "        data[i:i + block_size]\n",
    "        for i in range(0, usable, block_size)\n",
    "    ]\n",
    "\n",
    "    counts = Counter(blocks)\n",
    "\n",
    "    return {\n",
    "        \"blocks_checked\": len(blocks),\n",
    "        \"unique_blocks\": len(counts),\n",
    "        \"repeated_block_types\": sum(\n",
    "            count > 1 for count in counts.values()\n",
    "        ),\n",
    "        \"extra_duplicate_occurrences\": sum(\n",
    "            count - 1 for count in counts.values() if count > 1\n",
    "        )\n",
    "    }\n",
    "\n",
    "\n",
    "rows = []\n",
    "\n",
    "for file in files:\n",
    "    data = file.read_bytes()\n",
    "\n",
    "    _, byte_chi, byte_p = byte_frequency_test(data)\n",
    "    ones, zeros, bit_p = bit_balance_test(data)\n",
    "    runs, runs_p = runs_test(data)\n",
    "\n",
    "    rows.append({\n",
    "        \"File\": file.name,\n",
    "        \"Bytes\": len(data),\n",
    "        \"Byte chi-square\": byte_chi,\n",
    "        \"Byte p-value\": byte_p,\n",
    "        \"Bit p-value\": bit_p,\n",
    "        \"Runs\": runs,\n",
    "        \"Runs p-value\": runs_p,\n",
    "        \"Serial correlation\": serial_correlation(data),\n",
    "        \"Repeated patterns\": repeated_pattern_analysis(data)\n",
    "    })\n",
    "\n",
    "results = pd.DataFrame(rows)\n",
    "display(results)\n",
    "  "
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "272ba145",
   "metadata": {},
   "outputs": [],
   "source": [
    "\n",
    "## Interpretation\n",
    "\n",
    "1. **Byte-frequency chi-square test:** Compares the 256 observed\n",
    "byte counts with the expected equal counts. A small p-value may\n",
    "indicate a deviation from the uniform-byte model.\n",
    "\n",
    "2. **Bit-balance test:** Checks whether the number of one-bits is\n",
    "consistent with a probability of 0.5.\n",
    "\n",
    "3. **Runs test:** Checks whether the sequence of zero-bits and\n",
    "one-bits has a plausible number of consecutive runs.\n",
    "\n",
    "4. **Serial correlation:** Measures linear dependence between\n",
    "adjacent byte values. A value close to zero indicates little\n",
    "linear correlation at lag one.\n",
    "\n",
    "5. **Repeated patterns:** Counts repeated non-overlapping\n",
    "four-byte blocks. Some repetitions can occur by chance.\n",
    "\n",
    "A p-value above 0.05 means the test did not find statistically\n",
    "significant evidence against its particular null hypothesis at\n",
    "the 5% significance level. It does not prove that the data are\n",
    "random or cryptographically secure.\n",
    "\n",
    "The tests examine different properties and do not guarantee\n",
    "independence, unpredictability, correct entropy management, or\n",
    "resistance to an attacker.\n",
    "  "
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.15.1"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}